# Hojas de cálculo {#sec-spreadsheets}

## Introducción

Este capítulo te mostrará cómo trabajar con hojas de cálculo, por ejemplo archivos de Microsoft Excel, en Python. Ya vimos cómo importar archivos csv (y tsv) en @sec-data-import. En este capítulo te presentaremos herramientas para trabajar con datos en hojas de cálculo de Excel y Google Sheets.

Si tú o tus colaboradores usan hojas de cálculo para organizar datos que luego procesará una herramienta analítica como Python, te recomendamos leer el artículo "Data Organization in Spreadsheets" de Karl Broman y Kara Woo {cite}`broman2018data`. Las buenas prácticas que presenta este artículo te ahorrarán muchos dolores de cabeza más adelante, cuando importes los datos de una hoja de cálculo a Python para analizarlos y visualizarlos. (Para hojas de cálculo pensadas para que las lean personas, recomendamos el paquete [good practice tables](https://github.com/best-practice-and-impact/gptables).)

In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Requisitos previos

Para este capítulo necesitarás instalar el paquete **pandas**. También tendrás que instalar el paquete **openpyxl** ejecutando `uv add openpyxl` en la terminal.

## Leer archivos de Excel (y similares)

**pandas** puede leer archivos xls, xlsx, xlsm, xlsb, odf, ods y odt desde tu sistema de archivos local o desde una URL. También ofrece la opción de leer una sola hoja o una lista de hojas.

Para mostrar cómo funciona, trabajaremos con una hoja de cálculo de ejemplo llamada "students.xlsx". La figura siguiente muestra el aspecto de la hoja de cálculo.

![Vista de la hoja de cálculo students en Excel. La hoja de cálculo contiene información de 6 estudiantes: su ID, nombre completo, comida favorita, plan de comidas y edad.](https://github.com/hadley/r4ds/raw/main/screenshots/import-spreadsheets-students.png)

El primer argumento de `pd.read_excel()` es la ruta del archivo que se va a leer. Si has descargado el [archivo]() en tu computadora y lo has puesto en una subcarpeta llamada "data", deberías usar la ruta "data/students.xlsx", pero también podemos cargarlo directamente desde la URL.

In [ ]:
import pandas as pd

students = pd.read_excel(
    "https://github.com/aeturrell/python4DS/raw/main/data/students.xlsx"
)
students

Tenemos seis estudiantes en los datos y cinco variables para cada uno. Sin embargo, hay algunas cosas que quizá queramos corregir en este dataset:

- Los nombres de las columnas son muy desordenados. Puedes proporcionar nombres de columna que sigan un formato coherente; recomendamos `snake_case`, usando el argumento `names`.


In [ ]:
pd.read_excel(
    "https://github.com/aeturrell/python4DS/raw/main/data/students.xlsx",
    names=["student_id", "full_name", "favourite_food", "meal_plan", "age"],
)


- `age` se lee como una columna de objetos, pero en realidad debería ser numérica. Igual que con `read_csv()`, puedes pasar un argumento `dtype` a `read_excel()` y especificar los tipos de datos de las columnas que lees. Tus opciones incluyen `"boolean"`, `"int"`, `"float"`, `"datetime"`, `"string"` y más. Pero enseguida vemos que esto no va a funcionar con la columna "age", ya que mezcla números y texto: así que primero necesitamos convertir su texto en números.

In [ ]:
students = pd.read_excel(
    "data/students.xlsx",
    names=["student_id", "full_name", "favourite_food", "meal_plan", "age"],
)
students["age"] = students["age"].replace("five", 5)
students

Bien, ahora podemos aplicar los tipos de datos.

In [ ]:
students = students.astype(
    {
        "student_id": "Int64",
        "full_name": "string",
        "favourite_food": "string",
        "meal_plan": "category",
        "age": "Int64",
    }
)
students.info()

Hicieron falta varios pasos y algo de ensayo y error para cargar los datos exactamente en el formato que queremos, y esto no es inesperado. La ciencia de datos es un proceso iterativo. No hay forma de saber exactamente cómo serán los datos hasta que los cargas y les echas un vistazo. El patrón general que usamos es: cargar los datos, echarles un vistazo, ajustar el código, volver a cargarlos y repetir hasta que estés satisfecho con el resultado.

### Leer hojas individuales

Una característica importante que distingue las hojas de cálculo de los archivos planos es la noción de múltiples hojas. La figura siguiente muestra una hoja de cálculo de Excel con varias hojas. Los datos provienen del dataset **palmerpenguins** {cite}`horst2020palmerpenguins`. Cada hoja contiene información sobre pingüinos de una isla distinta donde se recogieron los datos.

![Vista de la hoja de cálculo penguins en Excel. La hoja de cálculo tiene tres hojas: Torgersen Island, Biscoe Island y Dream Island.](https://github.com/hadley/r4ds/raw/main/screenshots/import-spreadsheets-penguins-islands.png)

Puedes leer una sola hoja con el siguiente comando (para no mostrar el archivo completo, usaremos `.head()` para mostrar solo las primeras 5 filas):

In [ ]:
pd.read_excel(
    "https://github.com/aeturrell/python4DS/raw/main/data/penguins.xlsx",
    sheet_name="Torgersen Island",
).head()

Ahora bien, esto depende de que conozcamos de antemano los nombres de las hojas. Habrá situaciones en las que quieras leer datos sin mirar antes la hoja de cálculo de Excel. Para leer todas las hojas, usa `sheet_name=None`. El objeto que se crea es un diccionario con pares clave-valor que son, respectivamente, los nombres de las hojas y los dataframes. Veamos el segundo par clave-valor (ten en cuenta que debemos convertir los objetos keys() y values() en listas para luego obtener el segundo elemento de cada uno mediante un índice, es decir, `list(dictionary.keys())[<element number>]`).

Para que veas cómo funciona, primero imprimamos todas las claves obtenidas:

In [ ]:
penguins_dict = pd.read_excel(
    "https://github.com/aeturrell/python4DS/raw/main/data/penguins.xlsx",
    sheet_name=None,
)
print([x for x in penguins_dict.keys()])

Ahora mostremos el dataframe de la segunda entrada

In [ ]:
print(list(penguins_dict.keys())[1])
list(penguins_dict.values())[1].head()

Lo que realmente queremos es que estos tres datasets *coherentes* estén en un *mismo* dataframe. Para ello, podemos usar la función `pd.concat()`, que concatena cualquier iterable de dataframes.

In [ ]:
penguins = pd.concat(penguins_dict.values(), axis=0)
penguins

### Leer parte de una hoja

Como muchas personas usan las hojas de cálculo de Excel tanto para presentar como para almacenar datos, es bastante común encontrar en una hoja de cálculo celdas que no forman parte de los datos que quieres leer.

La figura siguiente muestra una hoja de cálculo así: en el centro de la hoja hay lo que parece un dataframe, pero hay texto ajeno en celdas por encima y por debajo de los datos.

![Vista de la hoja de cálculo deaths en Excel. La hoja de cálculo tiene cuatro filas en la parte superior que contienen información que no son datos; el texto 'For the same of consistency in the data layout, which is really a beautiful thing, I will keep making notes up here.' se reparte entre las celdas de esas cuatro filas superiores. Después hay un dataframe con información sobre las muertes de 10 personas famosas, incluidos sus nombres, profesiones, edades, si tienen hijos o no, y fechas de nacimiento y muerte. En la parte inferior hay otras cuatro filas de información que no son datos; el texto 'This has been really fun, but we're signing off now!' se reparte entre las celdas de esas cuatro filas inferiores.](https://github.com/hadley/r4ds/raw/main/screenshots/import-spreadsheets-deaths.png)



Puedes descargar esta hoja de cálculo desde [aquí](https://github.com/aeturrell/python4DS/tree/main/data) o cargarla directamente desde una URL. Si quieres cargarla desde el disco de tu computadora, primero tendrás que guardarla en una subcarpeta llamada "data".


Las tres filas superiores y las cuatro inferiores no forman parte del dataframe. Podríamos omitir las tres filas superiores con `skiprows`. Fíjate en que usamos `skiprows=4`, ya que la cuarta fila contiene los nombres de las columnas, no los datos.


In [ ]:
pd.read_excel("data/deaths.xlsx", skiprows=4)

También podríamos usar `nrows` para omitir las filas sobrantes de la parte inferior (otra opción sería omitir un número determinado de filas al final usando `skipfooter`).

In [ ]:
pd.read_excel("data/deaths.xlsx", skiprows=4, nrows=10)

## Tipos de datos

En los archivos CSV, todos los valores son strings. Esto no es especialmente fiel a los datos, pero es sencillo: todo es un string.

Los datos subyacentes en las hojas de cálculo de Excel son más complejos. Una celda puede ser una de estas cinco cosas:

-   Un valor lógico, como TRUE / FALSE

-   Un número, como "10" o "10.5"

-   Una fecha, que también puede incluir la hora, como "11/1/21" o "11/1/21 3:00 PM"

-   Un string, como "ten"

-   Una moneda, que admite valores numéricos en un rango limitado y cuatro dígitos decimales de precisión fija

Al trabajar con datos de hojas de cálculo, es importante tener en cuenta que la forma en que se almacenan los datos subyacentes puede ser muy distinta de lo que ves en la celda. Por ejemplo, Excel no tiene la noción de número entero. Todos los números se almacenan como de punto flotante (números reales), pero puedes elegir mostrar los datos con un número personalizable de decimales. Del mismo modo, las fechas en realidad se almacenan como números, concretamente el número de segundos transcurridos desde el 1 de enero de 1970. Puedes personalizar cómo se muestra la fecha aplicando formato en Excel. Para mayor confusión, también es posible tener algo que parece un número pero que en realidad es un string (por ejemplo, escribe `'10` en una celda de Excel).

Estas diferencias entre cómo se almacenan los datos subyacentes y cómo se muestran pueden causar sorpresas cuando los datos se cargan en herramientas analíticas como **pandas**. Por defecto, **pandas** intentará adivinar el tipo de datos de cada columna.
Un flujo de trabajo recomendable es dejar que **pandas** adivine inicialmente los tipos de las columnas, revisarlos y luego cambiar los tipos de datos que quieras.

## Escribir en Excel

Creemos un pequeño dataframe que luego podamos exportar. Fíjate en que `item` es una categoría y `quantity` es un entero.

In [ ]:
bake_sale = pd.DataFrame(
    {"item": pd.Categorical(["brownie", "cupcake", "cookie"]), "quantity": [10, 5, 8]}
)
bake_sale

Puedes volver a escribir los datos en disco como un archivo de Excel usando la función `<dataframe>.to_excel()`. El argumento de palabra clave `index=False` hace que solo se escriban las dos columnas, sin el índice que se añadió automáticamente en el paso anterior.

```python
bake_sale.to_excel("data/bake_sale.xlsx", index=False)
```

La figura siguiente muestra el aspecto de los datos en Excel.

![El dataframe de la venta de pasteles creado antes, visto en Excel.](https://github.com/hadley/r4ds/raw/main/screenshots/import-spreadsheets-bake-sale.png)

Al igual que al leer un CSV, la información sobre el tipo de dato se pierde cuando volvemos a leer los datos; puedes comprobarlo si vuelves a leer los datos y consultas `info` para ver los tipos de dato. Aunque conservamos `int64` porque **pandas** reconoce que la segunda columna era de tipo entero, perdimos el tipo de dato categórico de "item". Esta pérdida de tipos de dato hace que los archivos de Excel no sean fiables para guardar en caché resultados intermedios.

In [ ]:
pd.read_excel("data/bake_sale.xlsx").info()

### Salida con formato

Si necesitas más opciones de formato y más control sobre cómo escribes hojas de cálculo, consulta la documentación de [openpyxl](https://openpyxl.readthedocs.io/), que puede hacer prácticamente todo lo que te imagines. En general, publicar datos en hojas de cálculo no es la mejor opción; pero si de verdad quieres publicar datos en hojas de cálculo siguiendo las buenas prácticas, consulta [gptables](https://gptables.readthedocs.io/).